![image_1781187382053.png](./image_1781187382053.png "image_1781187382053.png")

![image_1781187399917.png](./image_1781187399917.png "image_1781187399917.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import Window

# Initialize Spark session
spark = SparkSession.builder.appName("CustomersOrders").getOrCreate()

# Create Customers DataFrame
customers_data = [
    (1, "Alice"),
    (2, "Bob"),
    (3, "Carol"),
    (4, "Dave")
]

customers_columns = ["id", "name"]

customers_df = spark.createDataFrame(customers_data, customers_columns)

# Create Orders DataFrame
orders_data = [
    (1, 1, 120, "2024-01-05"),
    (2, 1, 340, "2024-02-10"),
    (3, 1, 220, "2024-03-15"),
    (4, 2, 80, "2024-01-12"),
    (5, 2, 400, "2024-04-01"),
    (6, 3, 500, "2024-02-20"),
    (7, 3, 250, "2024-05-10"),
    (8, 4, 60, "2024-03-08")
]

orders_columns = ["id", "customer_id", "amount", "order_date"]

orders_df = spark.createDataFrame(orders_data, orders_columns)

# Show DataFrames
customers_df.show()
orders_df.show()


In [0]:
result_df = (
    customers_df.join(orders_df, customers_df.id == orders_df.customer_id)
    .groupBy(customers_df.id, customers_df.name)
    .agg(
        f.count(orders_df.id).alias("total_orders"),
        f.sum(orders_df.amount).alias("total_spent"),
        f.round(f.avg(orders_df.amount), 2).alias("avg_order_value"),
        f.min(orders_df.order_date).alias("first_order_date"),
        f.max(orders_df.order_date).alias("last_order_date"),
    )
    .select(
        f.col("id").alias("customer_id"),
        f.col("name").alias("customer_name"),
        f.col("total_orders"),
        f.col("total_spent"),
        f.col("avg_order_value"),
        f.col("first_order_date"),
        f.col("last_order_date"),
    )
    .orderBy(f.desc("total_spent"))
)
display(result_df)